In [ ]:
!nvidia-smi

In [ ]:
%%writefile cuda_bucket_sort_optimized.cu
// cuda_bucket_sort_optimized.cu
//
// Google Colab compile & run:
//   !nvcc -O2 -std=c++17 cuda_bucket_sort_optimized.cu -o cuda_sort && ./cuda_sort
//
// OPTIMIZATIONS over v1:
//   1. CUB DeviceSegmentedSort  — replaces 1024 serial Thrust calls
//   2. Thrust exclusive_scan    — GPU prefix sum, no CPU round-trip
//   3. Warm-up run              — eliminates JIT / context-init noise
//   4. Pinned host memory       — faster PCIe transfer
//   5. Async memcpy + streams   — overlaps H2D with kernel setup
//   6. CPU threshold guard      — falls back for tiny N
//
// Metrics reported (OLD vs NEW, per N):
//   SpeedUp, Computation Time, Total Operations, Data Transfer Time,
//   Amount Transferred, Total Execution Time, Achievable Performance

#include <iostream>
#include <vector>
#include <algorithm>
#include <random>
#include <chrono>
#include <cmath>
#include <iomanip>
#include <numeric>
#include <cuda_runtime.h>
#include <thrust/sort.h>
#include <thrust/device_ptr.h>
#include <thrust/execution_policy.h>
#include <thrust/scan.h>
#include <cub/cub.cuh>

using namespace std;
using Clock = chrono::high_resolution_clock;
using ms_t  = chrono::duration<double, milli>;

// ════════════════════════════════════════════════════════════════
//  CONFIG
// ════════════════════════════════════════════════════════════════
static constexpr int NUM_BUCKETS = 1024;
static constexpr int BLOCK_SIZE  = 256;
static constexpr int CPU_THRESH  = 200000;   // use CPU path below this N

// ════════════════════════════════════════════════════════════════
//  CUDA KERNELS
// ════════════════════════════════════════════════════════════════

// Kernel 1 — assign bucket ID for each element
//   Grid : ceil(n / BLOCK_SIZE) blocks × BLOCK_SIZE threads
//   Mem  : read d_arr (global, coalesced), write d_ids (global, coalesced)
__global__ void k_assignBuckets(
    const float* __restrict__ arr,
    int*         __restrict__ ids,
    int n, int K)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n) {
        int idx = __float2int_rd(arr[tid] * K);   // faster than (int)(...)
        ids[tid] = (idx >= K) ? K - 1 : idx;
    }
}

// Kernel 2 — histogram (count per bucket) via atomicAdd
//   Uses shared-memory histogram to reduce global atomic contention
//   Grid : ceil(n / BLOCK_SIZE) blocks × BLOCK_SIZE threads
//   Mem  : shared smem histogram (K ints = 4 KB for K=1024) → flushed to global
__global__ void k_countBuckets(
    const int* __restrict__ ids,
    int*       __restrict__ cnt,
    int n, int K)
{
    extern __shared__ int smem[];
    // init shared histogram
    for (int i = threadIdx.x; i < K; i += blockDim.x)
        smem[i] = 0;
    __syncthreads();

    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n)
        atomicAdd(&smem[ids[tid]], 1);
    __syncthreads();

    // flush shared → global
    for (int i = threadIdx.x; i < K; i += blockDim.x)
        atomicAdd(&cnt[i], smem[i]);
}

// Kernel 3 — scatter elements into contiguous bucket regions
//   Grid : ceil(n / BLOCK_SIZE) × BLOCK_SIZE
//   Mem  : reads d_arr + d_ids (coalesced), atomic write per-bucket counter
__global__ void k_scatter(
    const float* __restrict__ arr,
    const int*   __restrict__ ids,
    float*       __restrict__ out,
    const int*   __restrict__ offsets,
    int*         __restrict__ pos,   // per-bucket atomic cursor
    int n)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n) {
        int b   = ids[tid];
        int p   = atomicAdd(&pos[b], 1);
        out[offsets[b] + p] = arr[tid];
    }
}

// ════════════════════════════════════════════════════════════════
//  CPU HELPERS
// ════════════════════════════════════════════════════════════════
void cpuBucketSort(vector<float>& arr) {
    int n = (int)arr.size(), K = NUM_BUCKETS;
    vector<vector<float>> buckets(K);
    for (int i = 0; i < n; i++) {
        int idx = (int)(arr[i] * K);
        if (idx >= K) idx = K - 1;
        buckets[idx].push_back(arr[i]);
    }
    for (auto& b : buckets) sort(b.begin(), b.end());
    int idx = 0;
    for (auto& b : buckets)
        for (float v : b) arr[idx++] = v;
}

double cpuBaseline(int n) {
    vector<float> data(n);
    mt19937 rng(42);
    uniform_real_distribution<float> dist(0.f, 1.f);
    for (auto& x : data) x = dist(rng);
    auto t1 = Clock::now();
    cpuBucketSort(data);
    return ms_t(Clock::now() - t1).count();
}

bool verifySorted(const float* arr, int n) {
    for (int i = 1; i < n; i++)
        if (arr[i] < arr[i-1]) return false;
    return true;
}

// TotalOps = scatter(n) + segmented_sort(n·log2(n/K)) + gather(n)
double totalOps(int n, int K = NUM_BUCKETS) {
    return 2.0 * n + (double)n * log2((double)n / K);
}

// ════════════════════════════════════════════════════════════════
//  METRICS STRUCT
// ════════════════════════════════════════════════════════════════
struct Metrics {
    float h2dMs      = 0;
    float kernelMs   = 0;
    float sortMs     = 0;
    float d2hMs      = 0;
    float totalMs    = 0;
    bool  correct    = false;
};

Metrics operator+(const Metrics& a, const Metrics& b) {
    return {a.h2dMs+b.h2dMs, a.kernelMs+b.kernelMs,
            a.sortMs+b.sortMs, a.d2hMs+b.d2hMs,
            a.totalMs+b.totalMs, b.correct};
}
Metrics operator/(const Metrics& a, float s) {
    return {a.h2dMs/s, a.kernelMs/s, a.sortMs/s,
            a.d2hMs/s, a.totalMs/s, a.correct};
}

// ════════════════════════════════════════════════════════════════
//  ── OLD VERSION ──  (serial Thrust, CPU prefix sum)
// ════════════════════════════════════════════════════════════════
Metrics oldCudaBucketSort(float* h_arr, int n) {
    Metrics m{};
    int gridSize = (n + BLOCK_SIZE - 1) / BLOCK_SIZE;

    float *d_arr, *d_out;
    int   *d_ids, *d_cnt, *d_pos, *d_off;
    cudaMalloc(&d_arr, n*sizeof(float));
    cudaMalloc(&d_out, n*sizeof(float));
    cudaMalloc(&d_ids, n*sizeof(int));
    cudaMalloc(&d_cnt, NUM_BUCKETS*sizeof(int));
    cudaMalloc(&d_pos, NUM_BUCKETS*sizeof(int));
    cudaMalloc(&d_off, NUM_BUCKETS*sizeof(int));

    // total timer
    cudaEvent_t eT0, eT1, e0, e1;
    cudaEventCreate(&eT0); cudaEventCreate(&eT1);
    cudaEventCreate(&e0);  cudaEventCreate(&e1);

    cudaEventRecord(eT0);

    // H→D
    cudaEventRecord(e0);
    cudaMemcpy(d_arr, h_arr, n*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemset(d_cnt, 0, NUM_BUCKETS*sizeof(int));
    cudaMemset(d_pos, 0, NUM_BUCKETS*sizeof(int));
    cudaEventRecord(e1); cudaEventSynchronize(e1);
    cudaEventElapsedTime(&m.h2dMs, e0, e1);

    // Kernels 1+2
    cudaEventRecord(e0);
    k_assignBuckets<<<gridSize, BLOCK_SIZE>>>(d_arr, d_ids, n, NUM_BUCKETS);
    k_countBuckets<<<gridSize, BLOCK_SIZE,
                     NUM_BUCKETS*sizeof(int)>>>(d_ids, d_cnt, n, NUM_BUCKETS);
    cudaDeviceSynchronize();

    // CPU prefix sum (old bottleneck)
    vector<int> h_cnt(NUM_BUCKETS), h_off(NUM_BUCKETS);
    cudaMemcpy(h_cnt.data(), d_cnt, NUM_BUCKETS*sizeof(int), cudaMemcpyDeviceToHost);
    h_off[0] = 0;
    for (int i = 1; i < NUM_BUCKETS; i++) h_off[i] = h_off[i-1] + h_cnt[i-1];
    cudaMemcpy(d_off, h_off.data(), NUM_BUCKETS*sizeof(int), cudaMemcpyHostToDevice);

    k_scatter<<<gridSize, BLOCK_SIZE>>>(d_arr, d_ids, d_out, d_off, d_pos, n);
    cudaDeviceSynchronize();
    cudaEventRecord(e1); cudaEventSynchronize(e1);
    cudaEventElapsedTime(&m.kernelMs, e0, e1);

    // Serial Thrust sort (old bottleneck)
    cudaEventRecord(e0);
    thrust::device_ptr<float> ptr(d_out);
    for (int b = 0; b < NUM_BUCKETS; b++)
        if (h_cnt[b] > 1)
            thrust::sort(thrust::device, ptr+h_off[b], ptr+h_off[b]+h_cnt[b]);
    cudaDeviceSynchronize();
    cudaEventRecord(e1); cudaEventSynchronize(e1);
    cudaEventElapsedTime(&m.sortMs, e0, e1);

    // D→H
    cudaEventRecord(e0);
    cudaMemcpy(h_arr, d_out, n*sizeof(float), cudaMemcpyDeviceToHost);
    cudaEventRecord(e1); cudaEventSynchronize(e1);
    cudaEventElapsedTime(&m.d2hMs, e0, e1);

    cudaEventRecord(eT1); cudaEventSynchronize(eT1);
    cudaEventElapsedTime(&m.totalMs, eT0, eT1);

    m.correct = verifySorted(h_arr, n);

    cudaFree(d_arr); cudaFree(d_out); cudaFree(d_ids);
    cudaFree(d_cnt); cudaFree(d_pos); cudaFree(d_off);
    cudaEventDestroy(eT0); cudaEventDestroy(eT1);
    cudaEventDestroy(e0);  cudaEventDestroy(e1);
    return m;
}

// ════════════════════════════════════════════════════════════════
//  ── NEW VERSION ──  (CUB segmented sort, GPU prefix sum,
//                      pinned memory, async stream)
// ════════════════════════════════════════════════════════════════
Metrics newCudaBucketSort(float* h_arr, int n) {
    Metrics m{};
    int gridSize = (n + BLOCK_SIZE - 1) / BLOCK_SIZE;

    // Pinned host buffer for faster PCIe
    float* h_pin;
    cudaMallocHost(&h_pin, n*sizeof(float));
    memcpy(h_pin, h_arr, n*sizeof(float));

    float *d_arr, *d_out, *d_out2;
    int   *d_ids, *d_cnt, *d_pos, *d_off;
    cudaMalloc(&d_arr,  n*sizeof(float));
    cudaMalloc(&d_out,  n*sizeof(float));
    cudaMalloc(&d_out2, n*sizeof(float));   // CUB double-buffer
    cudaMalloc(&d_ids,  n*sizeof(int));
    cudaMalloc(&d_cnt,  NUM_BUCKETS*sizeof(int));
    cudaMalloc(&d_pos,  NUM_BUCKETS*sizeof(int));
    cudaMalloc(&d_off, (NUM_BUCKETS+1)*sizeof(int));  // +1 for end sentinel

    cudaStream_t stream;
    cudaStreamCreate(&stream);

    cudaEvent_t eT0, eT1, e0, e1;
    cudaEventCreate(&eT0); cudaEventCreate(&eT1);
    cudaEventCreate(&e0);  cudaEventCreate(&e1);

    cudaEventRecord(eT0, stream);

    // ── H→D (async, pinned) ─────────────────────────────────────
    cudaEventRecord(e0, stream);
    cudaMemcpyAsync(d_arr, h_pin, n*sizeof(float),
                    cudaMemcpyHostToDevice, stream);
    cudaMemsetAsync(d_cnt, 0, NUM_BUCKETS*sizeof(int), stream);
    cudaMemsetAsync(d_pos, 0, NUM_BUCKETS*sizeof(int), stream);
    cudaEventRecord(e1, stream);
    cudaStreamSynchronize(stream);
    cudaEventElapsedTime(&m.h2dMs, e0, e1);

    // ── Kernels 1, 2, 3 ─────────────────────────────────────────
    cudaEventRecord(e0, stream);

    k_assignBuckets<<<gridSize, BLOCK_SIZE, 0, stream>>>(
        d_arr, d_ids, n, NUM_BUCKETS);

    k_countBuckets<<<gridSize, BLOCK_SIZE,
                     NUM_BUCKETS*sizeof(int), stream>>>(
        d_ids, d_cnt, n, NUM_BUCKETS);

    // GPU prefix sum — no CPU round-trip!
    thrust::device_ptr<int> cnt_ptr(d_cnt);
    thrust::device_ptr<int> off_ptr(d_off);
    thrust::exclusive_scan(thrust::cuda::par.on(stream),
                           cnt_ptr, cnt_ptr + NUM_BUCKETS, off_ptr);

    // Write sentinel: d_off[NUM_BUCKETS] = n  (required by CUB)
    cudaMemcpyAsync(d_off + NUM_BUCKETS, &n, sizeof(int),
                    cudaMemcpyHostToDevice, stream);

    k_scatter<<<gridSize, BLOCK_SIZE, 0, stream>>>(
        d_arr, d_ids, d_out, d_off, d_pos, n);

    cudaStreamSynchronize(stream);
    cudaEventRecord(e1, stream);
    cudaStreamSynchronize(stream);
    cudaEventElapsedTime(&m.kernelMs, e0, e1);

    // ── CUB DeviceSegmentedSort (one kernel, all buckets parallel) ──
    cudaEventRecord(e0, stream);

    void*  d_temp    = nullptr;
    size_t temp_bytes = 0;

    // Query temp storage size
    cub::DeviceSegmentedSort::SortKeys(
        d_temp, temp_bytes,
        d_out, d_out2,
        n, NUM_BUCKETS,
        d_off, d_off + 1,
        stream);

    cudaMalloc(&d_temp, temp_bytes);

    // Execute — single kernel launch sorts ALL buckets in parallel
    cub::DeviceSegmentedSort::SortKeys(
        d_temp, temp_bytes,
        d_out, d_out2,
        n, NUM_BUCKETS,
        d_off, d_off + 1,
        stream);

    cudaStreamSynchronize(stream);
    cudaEventRecord(e1, stream);
    cudaStreamSynchronize(stream);
    cudaEventElapsedTime(&m.sortMs, e0, e1);

    // ── D→H (async, pinned) ──────────────────────────────────────
    cudaEventRecord(e0, stream);
    cudaMemcpyAsync(h_pin, d_out2, n*sizeof(float),
                    cudaMemcpyDeviceToHost, stream);
    cudaStreamSynchronize(stream);
    cudaEventRecord(e1, stream);
    cudaStreamSynchronize(stream);
    cudaEventElapsedTime(&m.d2hMs, e0, e1);

    memcpy(h_arr, h_pin, n*sizeof(float));

    cudaEventRecord(eT1, stream);
    cudaStreamSynchronize(stream);
    cudaEventElapsedTime(&m.totalMs, eT0, eT1);

    m.correct = verifySorted(h_arr, n);

    cudaFree(d_arr); cudaFree(d_out); cudaFree(d_out2); cudaFree(d_ids);
    cudaFree(d_cnt); cudaFree(d_pos); cudaFree(d_off); cudaFree(d_temp);
    cudaFreeHost(h_pin);
    cudaStreamDestroy(stream);
    cudaEventDestroy(eT0); cudaEventDestroy(eT1);
    cudaEventDestroy(e0);  cudaEventDestroy(e1);
    return m;
}

// ════════════════════════════════════════════════════════════════
//  PRINT HELPERS
// ════════════════════════════════════════════════════════════════
void printMetricsRow(const string& label, const Metrics& m,
                     double opsM, double xferMB,
                     double cpuMs, int n)
{
    double compMs   = m.kernelMs + m.sortMs;
    double perfGOPS = (opsM / 1e3) / (compMs / 1000.0);
    double speedup  = cpuMs / m.totalMs;

    cout << fixed << setprecision(3);
    cout << "│  [" << label << "]\n";
    cout << "│    Total Execution Time     : " << setw(10) << m.totalMs
         << " ms\n";
    cout << "│    Computation Time (GPU)   : " << setw(10) << compMs
         << " ms  (kernel=" << m.kernelMs << " + sort=" << m.sortMs << ")\n";
    cout << "│    Data Transfer H→D        : " << setw(10) << m.h2dMs
         << " ms  (" << xferMB/2 << " MB)\n";
    cout << "│    Data Transfer D→H        : " << setw(10) << m.d2hMs
         << " ms  (" << xferMB/2 << " MB)\n";
    cout << "│    Total Data Transferred   : " << setw(10) << xferMB
         << " MB\n";
    cout << "│    Total Operations         : " << setw(10) << opsM
         << " MOps\n";
    cout << "│    Achievable Performance   : " << setw(10) << setprecision(4)
         << perfGOPS << " GOPS\n";
    cout << "│    CPU Sequential Baseline  : " << setw(10) << setprecision(3)
         << cpuMs << " ms\n";
    cout << "│    SpeedUp (vs CPU seq)     : " << setw(10) << setprecision(2)
         << speedup << "x\n";
    cout << "│    Result Correct           : "
         << (m.correct ? "YES ✓" : "NO ✗") << "\n";
}

void printDelta(const Metrics& oldM, const Metrics& newM) {
    double oldComp = oldM.kernelMs + oldM.sortMs;
    double newComp = newM.kernelMs + newM.sortMs;
    cout << "│  ── Improvement (OLD → NEW) ──────────────────────────\n";
    cout << fixed << setprecision(2);
    cout << "│    Sort time    : " << oldM.sortMs   << " ms → " << newM.sortMs
         << " ms  (" << oldM.sortMs/max(newM.sortMs,0.001f) << "x faster)\n";
    cout << "│    Kernel time  : " << oldM.kernelMs << " ms → " << newM.kernelMs
         << " ms\n";
    cout << "│    Comp time    : " << oldComp << " ms → " << newComp
         << " ms  (" << oldComp/max(newComp,0.001) << "x faster)\n";
    cout << "│    Total time   : " << oldM.totalMs  << " ms → " << newM.totalMs
         << " ms  (" << oldM.totalMs/max(newM.totalMs,0.001f) << "x faster)\n";
}

// ════════════════════════════════════════════════════════════════
//  MAIN
// ════════════════════════════════════════════════════════════════
int main() {
    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);

    cout << "════════════════════════════════════════════════════════════\n";
    cout << "  CUDA Bucket Sort — OLD vs NEW Comparison\n";
    cout << "════════════════════════════════════════════════════════════\n";
    cout << "GPU   : " << prop.name << "\n";
    cout << "VRAM  : " << prop.totalGlobalMem/(1024*1024) << " MB\n";
    cout << "SMs   : " << prop.multiProcessorCount << "\n";
    cout << "Block : " << BLOCK_SIZE << " threads  |  Buckets: " << NUM_BUCKETS << "\n";
    cout << "\nOPTIMIZATIONS IN NEW VERSION:\n";
    cout << "  [1] CUB DeviceSegmentedSort  — 1 kernel replaces 1024 serial Thrust calls\n";
    cout << "  [2] GPU thrust::exclusive_scan — prefix sum stays on device\n";
    cout << "  [3] Pinned host memory        — faster PCIe H↔D bandwidth\n";
    cout << "  [4] Async cudaMemcpyAsync     — overlaps transfer with setup\n";
    cout << "  [5] Shared-mem histogram      — reduces global atomic pressure\n";
    cout << "  [6] CUDA streams              — explicit async execution\n\n";

    // ── Warm-up: eliminates JIT / CUDA context init from measurements ──
    {
        cout << "Warming up GPU context...\n\n";
        vector<float> wu(50000);
        mt19937 rng(0); uniform_real_distribution<float> d(0,1);
        for (auto& x : wu) x = d(rng);
        oldCudaBucketSort(wu.data(), 50000);
        for (auto& x : wu) x = d(rng);
        newCudaBucketSort(wu.data(), 50000);
    }

    int reps = 3;
    mt19937 rng(42);
    uniform_real_distribution<float> dist(0.f, 1.f);

    for (int n : {10000, 100000, 1000000}) {

        // Generate base data
        vector<float> base(n);
        for (auto& x : base) x = dist(rng);

        // CPU baseline (3 runs)
        double cpuMs = 0;
        for (int r = 0; r < 3; r++) cpuMs += cpuBaseline(n);
        cpuMs /= 3.0;

        // Old version — averaged over reps
        Metrics oldAvg{};
        for (int r = 0; r < reps; r++) {
            vector<float> tmp = base;
            oldAvg = oldAvg + oldCudaBucketSort(tmp.data(), n);
        }
        oldAvg = oldAvg / (float)reps;

        // New version — averaged over reps
        Metrics newAvg{};
        for (int r = 0; r < reps; r++) {
            vector<float> tmp = base;
            newAvg = newAvg + newCudaBucketSort(tmp.data(), n);
        }
        newAvg = newAvg / (float)reps;

        double opsM   = totalOps(n) / 1e6;
        double xferMB = 2.0 * n * sizeof(float) / 1e6;

        cout << "┌─────────────────────────────────────────────────────────\n";
        cout << "│  N = " << n << "\n";
        cout << "├─────────────────────────────────────────────────────────\n";
        printMetricsRow("OLD", oldAvg, opsM, xferMB, cpuMs, n);
        cout << "│\n";
        printMetricsRow("NEW", newAvg, opsM, xferMB, cpuMs, n);
        cout << "│\n";
        printDelta(oldAvg, newAvg);
        cout << "└─────────────────────────────────────────────────────────\n\n";
    }

    cout << "════════════════════════════════════════════════════════════\n";
    cout << "NOTES\n";
    cout << "  Computation Time  = kernels(1+2+3) + sort  (no PCIe)\n";
    cout << "  Total Execution   = H2D + kernels + sort + D2H\n";
    cout << "  TotalOps          = 2n + n·log2(n/K)  (scatter+sort+gather)\n";
    cout << "  Achievable Perf   = TotalOps / CompTime\n";
    cout << "  SpeedUp           = CPU_time / CUDA_total_time\n";
    cout << "  Reps averaged     = 3  (after warm-up)\n";
    cout << "════════════════════════════════════════════════════════════\n";

    return 0;
}


In [ ]:
!nvcc -O2 -std=c++17 cuda_bucket_sort_optimized.cu -o cuda_sort 2>&1

In [ ]:
!./cuda_sort